# Lab 02.2 — Formati di file e Apache Parquet

Spark legge e scrive molti formati con la stessa API (`spark.read.format(...)`, `df.write.format(...)`).
I formati si dividono in due famiglie:

| | Orientati alle **righe** | Orientati alle **colonne** |
|---|---|---|
| Esempi | CSV, JSON, Avro | **Parquet**, ORC |
| Schema | assente (CSV) o ripetuto (JSON) | salvato nei metadati |
| Lettura di poche colonne | legge tutto il file | legge solo le colonne utili |
| Compressione | scarsa | ottima (valori simili vicini) |

Parquet organizza i dati in **row group** → **column chunk** → **pagine**, e memorizza per ogni chunk statistiche (min/max, null) che permettono di **saltare** blocchi interi.

In [ ]:
import os, time, glob, shutil
from pyspark.sql import SparkSession, functions as F
import dati_shop

spark = (SparkSession.builder
         .appName("Lab02-Parquet")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.sql.shuffle.partitions", "8")
         # cartella in cui il catalogo salva le tabelle gestite (sezione 9)
         .config("spark.sql.warehouse.dir", "/home/jovyan/data/warehouse")
         .getOrCreate())

OUT = "/home/jovyan/data/formati"
shutil.rmtree(OUT, ignore_errors=True)

def dimensione(path):
    """Dimensione su disco (MB) di una cartella scritta da Spark."""
    return sum(os.path.getsize(f) for f in glob.glob(f"{path}/**/*", recursive=True) if os.path.isfile(f)) / 1e6

ordini = dati_shop.carica(spark)["ordini"].cache()
print(f"{ordini.count():,} ordini")

## 1. Scrittura dello stesso DataFrame in formati diversi
Ogni partizione del DataFrame viene scritta in parallelo da un executor in un file `part-XXXXX`.

In [ ]:
formati = {"csv": {"header": True}, "json": {}, "orc": {}, "parquet": {}}
for fmt, opzioni in formati.items():
    t0 = time.time()
    ordini.write.format(fmt).options(**opzioni).mode("overwrite").save(f"{OUT}/ordini_{fmt}")
    print(f"{fmt:8s} scritto in {time.time() - t0:5.2f}s  →  {dimensione(f'{OUT}/ordini_{fmt}'):7.1f} MB")

In [ ]:
# Un file per partizione + _SUCCESS (marcatore di scrittura completata); i .crc sono checksum locali
print(sorted(f for f in os.listdir(f"{OUT}/ordini_parquet") if not f.endswith(".crc")))

## 2. Schema: inferito vs memorizzato
Il CSV non contiene tipi: Spark deve leggere i dati per indovinarli (`inferSchema`) oppure riceverli dall'utente.
Parquet salva lo schema nel footer del file: la lettura è immediata e i tipi sono esatti.

In [ ]:
spark.read.option("header", True).csv(f"{OUT}/ordini_csv").printSchema()            # tutto stringa
spark.read.option("header", True).option("inferSchema", True).csv(f"{OUT}/ordini_csv").printSchema()
spark.read.parquet(f"{OUT}/ordini_parquet").printSchema()                            # tipi originali

## 3. Lettura colonnare: column pruning
Una query che usa **una sola colonna** in Parquet legge solo quella; in CSV/JSON deve analizzare ogni riga intera.

In [ ]:
def misura(df, etichetta):
    t0 = time.time()
    df.select(F.avg("quantita")).collect()
    print(f"{etichetta:8s}: {time.time() - t0:.2f}s")

schema = ordini.schema   # schema fornito → niente inferenza, confronto più equo
misura(spark.read.schema(schema).option("header", True).csv(f"{OUT}/ordini_csv"), "csv")
misura(spark.read.schema(schema).json(f"{OUT}/ordini_json"), "json")
misura(spark.read.orc(f"{OUT}/ordini_orc"), "orc")
misura(spark.read.parquet(f"{OUT}/ordini_parquet"), "parquet")

## 4. Dentro un file Parquet
Con PyArrow leggiamo i metadati: row group, colonne, codifica, compressione e **statistiche min/max**.

In [ ]:
import pyarrow.parquet as pq

file0 = sorted(glob.glob(f"{OUT}/ordini_parquet/part-*.parquet"))[0]
meta = pq.ParquetFile(file0).metadata
print(f"File: {os.path.basename(file0)}\nrighe: {meta.num_rows:,}  row group: {meta.num_row_groups}  colonne: {meta.num_columns}")
print(pq.ParquetFile(file0).schema_arrow)

rg = meta.row_group(0)
print(f"\n{'colonna':12s} {'compressione':12s} {'byte':>10s}  min → max")
for i in range(rg.num_columns):
    col = rg.column(i)
    st = col.statistics
    print(f"{col.path_in_schema:12s} {col.compression:12s} {col.total_compressed_size:10,d}  {st.min} → {st.max}")

## 5. Predicate pushdown
Il filtro viene "spinto" nel lettore Parquet (`PushedFilters`): i row group le cui statistiche min/max escludono il valore cercato non vengono nemmeno letti.

In [ ]:
pq_ordini = spark.read.parquet(f"{OUT}/ordini_parquet")
q = pq_ordini.where((F.col("ordine_id") < 1000) & (F.col("canale") == "app")).select("ordine_id", "data")
q.explain()
print("Righe trovate:", q.count())

## 6. Partizionamento su disco (`partitionBy`)
Con `partitionBy` i dati vengono organizzati in sottocartelle `colonna=valore`.
Le query che filtrano su quelle colonne leggono **solo le cartelle necessarie** (*partition pruning*).

In [ ]:
(ordini.withColumn("anno", F.year("data")).withColumn("mese", F.month("data"))
       .write.partitionBy("anno", "mese").mode("overwrite").parquet(f"{OUT}/ordini_partizionati"))

anni = sorted(d for d in os.listdir(f"{OUT}/ordini_partizionati") if d.startswith("anno="))
for d in anni:
    mesi = sorted((m for m in os.listdir(f"{OUT}/ordini_partizionati/{d}") if m.startswith("mese=")), key=lambda m: int(m[5:]))
    print(d, "→", mesi[:4], "...")

In [ ]:
part = spark.read.parquet(f"{OUT}/ordini_partizionati")
q = part.where("anno = 2025 AND mese = 3").groupBy("canale").count()
q.explain()        # PartitionFilters: [anno = 2025, mese = 3]
q.show()

In [ ]:
# Confronto: la stessa query con e senza partizionamento
t0 = time.time(); part.where("anno = 2025 AND mese = 3").count(); t_part = time.time() - t0
t0 = time.time(); pq_ordini.where("year(data) = 2025 AND month(data) = 3").count(); t_full = time.time() - t0
print(f"partizionato: {t_part:.2f}s   non partizionato: {t_full:.2f}s")

## 7. Codec di compressione
Parquet supporta diversi codec: `snappy` (default, veloce), `zstd` (ottimo compromesso), `gzip` (compatto ma lento), `none`.

In [ ]:
for codec in ["none", "snappy", "gzip", "zstd"]:
    path = f"{OUT}/codec_{codec}"
    t0 = time.time()
    ordini.write.option("compression", codec).mode("overwrite").parquet(path)
    print(f"{codec:7s}: {dimensione(path):6.1f} MB  scrittura {time.time() - t0:.2f}s")

## 8. Evoluzione dello schema
Nel tempo un dataset può acquisire nuove colonne. Con `mergeSchema` Spark unisce gli schemi dei diversi file; le righe vecchie avranno `NULL` nelle colonne nuove.

In [ ]:
v1 = spark.createDataFrame([(1, "Alice"), (2, "Bob")], "id INT, nome STRING")
v2 = spark.createDataFrame([(3, "Carla", "carla@example.com")], "id INT, nome STRING, email STRING")
v1.write.mode("overwrite").parquet(f"{OUT}/utenti/versione=1")
v2.write.mode("overwrite").parquet(f"{OUT}/utenti/versione=2")

spark.read.option("mergeSchema", True).parquet(f"{OUT}/utenti").orderBy("id").show()

## 9. SQL direttamente sui file e tabelle del catalogo

In [ ]:
# Interrogare un file senza registrare nulla
spark.sql(f"SELECT canale, COUNT(*) AS n FROM parquet.`{OUT}/ordini_parquet` GROUP BY canale").show()

In [ ]:
# Tabella gestita dal catalogo di Spark (file Parquet nella cartella "warehouse").
# Nota: senza un metastore (es. Hive) il catalogo vive in memoria e la definizione si perde a fine sessione.
spark.sql("CREATE DATABASE IF NOT EXISTS shop")
(ordini.where("canale = 'negozio'")
       .write.mode("overwrite").format("parquet").saveAsTable("shop.ordini_negozio"))
spark.sql("SHOW TABLES IN shop").show()
spark.sql("DESCRIBE TABLE EXTENDED shop.ordini_negozio").where("col_name IN ('Location','Provider','Type')").show(truncate=False)
spark.table("shop.ordini_negozio").count()

In [ ]:
spark.stop()